NN and Device Setup

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Optuna Setup

In [3]:
import optuna

X = torch.tensor(
    [[0., 0.],
     [0., 1.],
     [1., 0.],
     [1., 1.]],
    device=device
)

y = torch.tensor(
    [[0.],
     [0.],
     [0.],
     [1.]],
    device=device
)

def build_model(n_layers, hidden_units, activation, bias):
    layers = []
    in_dim = 2

    act_map = {
        "relu": nn.ReLU,
        "tanh": nn.Tanh,
        "sigmoid": nn.Sigmoid,
    }

    for _ in range(n_layers):
        layers.append(nn.Linear(in_dim, hidden_units, bias=bias))
        layers.append(act_map[activation]())
        in_dim = hidden_units

    layers.append(nn.Linear(in_dim, 1, bias=bias))
    return nn.Sequential(*layers).to(device)

def objective(trial):
    n_layers = trial.suggest_int("n_layers", 0, 2)
    hidden_units = trial.suggest_int("hidden_units", 1, 8)
    activation = trial.suggest_categorical("activation", ["relu", "tanh", "sigmoid"])
    bias = trial.suggest_categorical("bias", [True, False])

    lr = trial.suggest_float("lr", 1e-4, 1.0, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["SGD", "Adam"])

    model = build_model(n_layers, hidden_units, activation, bias)

    optimizer = (
        optim.Adam(model.parameters(), lr=lr)
        if optimizer_name == "Adam"
        else optim.SGD(model.parameters(), lr=lr)
    )

    loss_fn = nn.BCEWithLogitsLoss()

    for _ in range(200):
        optimizer.zero_grad()
        logits = model(X)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        preds = (torch.sigmoid(model(X)) > 0.5).float()
        acc = (preds == y).float().mean().item()

    return acc

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best accuracy:", study.best_value)
print("Best parameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


c:\Users\Mervyn Rangasamy\Documents\2026\COS 700\Repo\COS700\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-02-18 08:16:49,698] A new study created in memory with name: no-name-940d415d-2913-46ba-95dc-cba3af4932dc
[I 2026-02-18 08:16:54,378] Trial 0 finished with value: 0.5 and parameters: {'n_layers': 1, 'hidden_units': 6, 'activation': 'relu', 'bias': False, 'lr': 0.0007648374575626255, 'optimizer': 'SGD'}. Best is trial 0 with value: 0.5.
[I 2026-02-18 08:16:54,794] Trial 1 finished with value: 0.75 and parameters: {'n_layers': 2, 'hidden_units': 8, 'activation': 'sigmoid', 'bias': False, 'lr': 0.0020188754842952947, 'optimizer': 'SGD'}. Best is trial 1 with value: 0.75.
[I 2026-02-18 08:16:55,196] Trial 2 finished with value: 0.75 and parameters: {'n_layers': 2, 'hidden_units': 5, 'activation': '

Best accuracy: 1.0
Best parameters:
  n_layers: 1
  hidden_units: 4
  activation: relu
  bias: False
  lr: 0.023600361859255438
  optimizer: Adam


EvoTorch

In [ ]:
from evotorch import Problem
from evotorch.algorithms import PGPE
from evotorch.logging import StdOutLogger

# Genome definition
# [n_layers, hidden_units, activation_id, bias, optimizer_id, log10_lr]
# n_layers        ∈ {0,1,2}
# hidden_units    ∈ [1,8]
# activation_id   ∈ {0=relu,1=tanh,2=sigmoid}
# bias            ∈ {0=False,1=True}
# optimizer_id    ∈ {0=SGD,1=Adam}
# log10_lr        ∈ [-4, 0]

GENOME_DIM = 6

def build_model(n_layers, hidden_units, activation, bias):
    layers = []
    in_dim = 2

    act_map = {
        0: nn.ReLU,
        1: nn.Tanh,
        2: nn.Sigmoid,
    }

    for _ in range(n_layers):
        layers.append(nn.Linear(in_dim, hidden_units, bias=bias))
        layers.append(act_map[activation]())
        in_dim = hidden_units

    layers.append(nn.Linear(in_dim, 1, bias=bias))
    return nn.Sequential(*layers).to(device)

def fitness_fn(genomes: torch.Tensor) -> torch.Tensor:

    if genomes.ndim == 1:
        genomes = genomes.unsqueeze(0)

    fitness = torch.zeros(genomes.shape[0], device=genomes.device)

    for i,g in enumerate(genomes):
        n_layers = int(torch.clamp(torch.round(g[0]), 0, 2).item())
        hidden_units = int(torch.clamp(torch.round(g[1]), 1, 8).item())
        activation = int(torch.clamp(torch.round(g[2]), 0, 2).item())
        bias = bool(torch.clamp(torch.round(g[3]), 0, 1).item())
        optimizer_id = int(torch.clamp(torch.round(g[4]), 0, 1).item())
        lr = 10 ** torch.clamp(g[5], -4, 0).item()

        model = build_model(n_layers, hidden_units, activation, bias)

        optimizer = (
            optim.Adam(model.parameters(), lr=lr)
            if optimizer_id == 1
            else optim.SGD(model.parameters(), lr=lr)
        )

        loss_fn = nn.BCEWithLogitsLoss()

        for _ in range(200):
            optimizer.zero_grad()
            logits = model(X)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            preds = (torch.sigmoid(model(X)) > 0.5).float()
            acc = (preds == y).float().mean().item()

        fitness[i] = acc

    return torch.tensor(fitness, device=device)

problem = Problem(
    "max",
    fitness_fn,
    solution_length=GENOME_DIM,
    device=device,
    initial_bounds=(
        torch.tensor([0, 1, 0, 0, 0, -4.0], device=device),
        torch.tensor([2, 8, 2, 1, 1, 0.0], device=device),
    ),
)

searcher = PGPE(
    problem,
    popsize=32,
    radius_init=1.0,
    center_learning_rate=0.15,
    stdev_learning_rate=0.1,
)

StdOutLogger(searcher)

searcher.run(50)

best = searcher.get_status_value["best"]
print("\nBest genome :", best.values)
print("\nBest Fitness : ", best.evaluation)



[2026-02-18 10:54:57] INFO     <23560> evotorch.core: Instance of `Problem` (id:2827211968624) -- The `dtype` for the problem's decision variables is set as torch.float32
[2026-02-18 10:54:57] INFO     <23560> evotorch.core: Instance of `Problem` (id:2827211968624) -- `eval_dtype` (the dtype of the fitnesses and evaluation data) is set as torch.float32
[2026-02-18 10:54:57] INFO     <23560> evotorch.core: Instance of `Problem` (id:2827211968624) -- The `device` of the problem is set as cuda
[2026-02-18 10:54:57] INFO     <23560> evotorch.core: Instance of `Problem` (id:2827211968624) -- The number of actors that will be allocated for parallelized evaluation is 0
[2026-02-18 10:54:57] INFO     <23560> evotorch.optimizers: Instance of `ClipUp` (id:2827211867632) -- The maximum speed for the ClipUp optimizer is set as 0.3 which is two times the given step size.


c:\Users\Mervyn Rangasamy\Documents\2026\COS 700\Repo\COS700\venv\Lib\site-packages\evotorch\tools\misc.py:1201: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  result = torch.tensor(data, **kwargs)
C:\Users\Mervyn Rangasamy\AppData\Local\Temp\ipykernel_23560\3698997550.py:72: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(fitness, device=device)


         iter : 1
    mean_eval : 0.59375
  median_eval : 0.5
pop_best_eval : 1.0

         iter : 2
    mean_eval : 0.625
  median_eval : 0.75
pop_best_eval : 0.75

         iter : 3
    mean_eval : 0.6015625
  median_eval : 0.5
pop_best_eval : 0.75

         iter : 4
    mean_eval : 0.6171875
  median_eval : 0.5
pop_best_eval : 0.75

         iter : 5
    mean_eval : 0.625
  median_eval : 0.75
pop_best_eval : 0.75

         iter : 6
    mean_eval : 0.6484375
  median_eval : 0.75
pop_best_eval : 1.0

         iter : 7
    mean_eval : 0.6640625
  median_eval : 0.75
pop_best_eval : 1.0

         iter : 8
    mean_eval : 0.7109375
  median_eval : 0.75
pop_best_eval : 0.75

         iter : 9
    mean_eval : 0.6796875
  median_eval : 0.75
pop_best_eval : 1.0

         iter : 10
    mean_eval : 0.671875
  median_eval : 0.75
pop_best_eval : 1.0

         iter : 11
    mean_eval : 0.7265625
  median_eval : 0.75
pop_best_eval : 1.0

         iter : 12
    mean_eval : 0.7109375
  median_eval : 

AttributeError: 'Problem' object has no attribute 'best_solution'

: 